# DataSet Fetching

### General description :
This Jupiter Notebook is the code to get the dataset I do my analysis on. It is my data as a Super Smash Bros Ultimate (SSBU) player/competitor.<br>
I got it on the startgg website which is the site where most of the SSBU tournaments are created and recorded.<br>
I will focus my analysis on my data as a player.
### API
Anyone can get an API for the whole dataset of all the tournaments that are created on startgg (not only SSBU but also other games).<br>
To get an API, you just need to create an account, go into the developer setting and create your own personal API.<br>
In this code I use my personal API token which won't be active after I finish my project. Please use your own if you want to get data from the startgg database.<br>

## Requirements

**Python packages:**
```bash
pip install requests
```

**Before running:**
- Your own start.gg API token, set as `API_TOKEN`.
- The target player's numeric `player.id` (not their profile URL slug), set as `PLAYER_ID`.

### Config

In [1]:
import requests, json, time, csv
from datetime import datetime, timedelta

API_TOKEN = "7a6f7b8db38f760bb210062b7ae25959" #This is my API token
URL = "https://api.start.gg/gql/alpha"
HEADERS = {"Authorization": f"Bearer {API_TOKEN}", "Content-Type": "application/json"}

### Find the data of the player Sumbra

In [2]:
LOOKUP_QUERY = """
query GetUserPlayerId($slug: String!) {
  user(slug: $slug) {
    id
    player {
      id
      gamerTag
    }
  }
}
"""

resp = requests.post(URL, headers=HEADERS, json={
    "query": LOOKUP_QUERY,
    "variables": {"slug": "user/393dd719"}
})
data = resp.json()
print(json.dumps(data, indent=2))

{
  "data": {
    "user": {
      "id": 1819892,
      "player": {
        "id": 2742918,
        "gamerTag": "Sumbra"
      }
    }
  },
  "extensions": {
    "cacheControl": {
      "version": 1,
      "hints": null
    },
    "queryComplexity": 2
  },
  "actionRecords": []
}


### Query

In [21]:
PLAYER_ID = 2742918  # Filled manually

QUERY = """
query PlayerSets($playerId: ID!, $page: Int!) {
  player(id: $playerId) {
    id
    gamerTag
    sets(page: $page, perPage: 25, filters: {}) {
      pageInfo { totalPages }
      nodes {
        id
        displayScore
        winnerId
        fullRoundText
        event {
          id
          slug
          name
          startAt
          numEntrants
          tournament { name countryCode city lat lng }
        }
        slots {
          entrant {
            id
            name
            seeds { seedNum }
            participants { player { id gamerTag } }
          }
        }
        games {
          winnerId
          selections {
            entrant { id }
            selectionValue
            character { name }
          }
        }
      }
    }
  }
}
"""

### Fetch function

In [23]:
def fetch_all_sets(player_id):
    all_sets = []
    page = 1
    while True:
        resp = requests.post(URL, headers=HEADERS, json={
            "query": QUERY,
            "variables": {"playerId": player_id, "page": page}
        })
        data = resp.json()
        if "errors" in data:
            print(data["errors"])
            break
        sets_data = data["data"]["player"]["sets"]
        all_sets.extend(sets_data["nodes"])
        total_pages = sets_data["pageInfo"]["totalPages"]
        print(f"Fetched page {page}/{total_pages}")
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.7)  # stay under rate limits
    return all_sets

sets = fetch_all_sets(PLAYER_ID)
len(sets)  # sanity check

Fetched page 1/36
Fetched page 2/36
Fetched page 3/36
Fetched page 4/36
Fetched page 5/36
Fetched page 6/36
Fetched page 7/36
Fetched page 8/36
Fetched page 9/36
Fetched page 10/36
Fetched page 11/36
Fetched page 12/36
Fetched page 13/36
Fetched page 14/36
Fetched page 15/36
Fetched page 16/36
Fetched page 17/36
Fetched page 18/36
Fetched page 19/36
Fetched page 20/36
Fetched page 21/36
Fetched page 22/36
Fetched page 23/36
Fetched page 24/36
Fetched page 25/36
Fetched page 26/36
Fetched page 27/36
Fetched page 28/36
Fetched page 29/36
Fetched page 30/36
Fetched page 31/36
Fetched page 32/36
Fetched page 33/36
Fetched page 34/36
Fetched page 35/36
Fetched page 36/36


889

### Filter for player Sumbra since August 1st 2023

In [24]:
from datetime import datetime

cutoff = datetime(2023, 8, 1)

filtered = [
    s for s in sets
    if s["event"]["startAt"] and datetime.fromtimestamp(s["event"]["startAt"]) > cutoff
]

print(f"{len(filtered)} sets since August 1st, 2023")

841 sets since August 1st, 2023


In [25]:
filtered[0]

{'id': 109260248,
 'displayScore': 'Sumbra 0 - greenmontanafan75 2',
 'winnerId': 25043044,
 'fullRoundText': 'Losers Round 5',
 'event': {'id': 1732101,
  'slug': 'tournament/dawnpath-s2-02/event/ultimate-singles',
  'name': 'Ultimate Singles',
  'startAt': 1792778400,
  'numEntrants': 64,
  'tournament': {'name': 'Dawnpath S2 #02',
   'countryCode': 'FR',
   'city': 'Le Kremlin-Bicêtre',
   'lat': 48.81566369999999,
   'lng': 2.362835}},
 'slots': [{'entrant': {'id': 25078917,
    'name': 'Sumbra',
    'seeds': [{'seedNum': 11}],
    'participants': [{'player': {'id': 2742918, 'gamerTag': 'Sumbra'}}]}},
  {'entrant': {'id': 25043044,
    'name': 'greenmontanafan75',
    'seeds': [{'seedNum': 14}],
    'participants': [{'player': {'id': 2839148,
       'gamerTag': 'greenmontanafan75'}}]}}],
 'games': [{'winnerId': 25043044,
   'selections': [{'entrant': {'id': 25043044},
     'selectionValue': 1302,
     'character': {'name': 'Mario'}},
    {'entrant': {'id': 25078917},
     'selectio

### Creation of the csv

In [26]:
HEADER = [
    "tournament", "event", "date", "round",
    "my_score", "opponent_score", "my_seed", "opponent_seed",
    "opponent", "won", "my_characters", "opponent_characters",
    # new columns
    "set_id", "event_id", "event_slug", "num_entrants",
    "country", "city", "lat", "lng",
]

def is_me(slot):
    return any(p["player"]["id"] == PLAYER_ID for p in slot["entrant"]["participants"])

def first_seed(slot):
    if slot and slot["entrant"]["seeds"]:
        return slot["entrant"]["seeds"][0]["seedNum"]
    return "unreported"

with open("player_sets.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(HEADER)

    for s in filtered:
        ev = s["event"]
        tour = ev["tournament"]
        date = datetime.fromtimestamp(ev["startAt"]).strftime("%Y-%m-%d")

        my_slot = next((slot for slot in s["slots"] if is_me(slot)), None)
        opp_slot = next((slot for slot in s["slots"] if not is_me(slot)), None)
        my_id = my_slot["entrant"]["id"] if my_slot else None
        opp_id = opp_slot["entrant"]["id"] if opp_slot else None

        opponents = [p["player"]["gamerTag"]
                     for p in opp_slot["entrant"]["participants"]] if opp_slot else []

        games = s.get("games") or []
        my_score = sum(1 for g in games if g.get("winnerId") == my_id)
        opponent_score = sum(1 for g in games if g.get("winnerId") == opp_id)

        my_chars, opp_chars = set(), set()
        for g in games:
            for sel in (g.get("selections") or []):
                if not sel.get("character"):
                    continue
                if sel["entrant"]["id"] == my_id:
                    my_chars.add(sel["character"]["name"])
                elif sel["entrant"]["id"] == opp_id:
                    opp_chars.add(sel["character"]["name"])

        writer.writerow([
            tour["name"], ev["name"], date, s["fullRoundText"],
            my_score, opponent_score, first_seed(my_slot), first_seed(opp_slot),
            ", ".join(opponents), s["winnerId"] == my_id,
            ", ".join(my_chars) if my_chars else "Greninja",
            ", ".join(opp_chars) if opp_chars else "unreported",
            # new columns (empty string when the API returns null, e.g. online events)
            s["id"], ev["id"], ev.get("slug") or "", ev.get("numEntrants") or "",
            tour.get("countryCode") or "", tour.get("city") or "",
            tour.get("lat") or "", tour.get("lng") or "",
        ])

print("Saved to player_sets.csv")

Saved to player_sets.csv


In [27]:
import pandas as pd
chk = pd.read_csv("player_sets.csv")
print(chk.shape)
print(chk[["num_entrants", "country", "city", "lat", "lng"]].isna().mean().round(2))
print(chk["country"].value_counts())

(841, 20)
num_entrants    0.00
country         0.01
city            0.01
lat             0.01
lng             0.01
dtype: float64
country
FR    677
CA    156
Name: count, dtype: int64
